<a href="https://colab.research.google.com/github/ainiowais111-sketch/My-QloRA-Project/blob/main/My_QloRA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -U transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attemptin

In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="train.jsonl",
    split="train"
)

print(dataset)
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'response'],
    num_rows: 5
})
{'instruction': 'What is Artificial Intelligence?', 'response': 'Artificial Intelligence is a field of computer science that focuses on creating machines and systems that can perform tasks that normally require human intelligence.'}


In [5]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Model
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit QLoRA configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False

print("✅ Model loaded successfully!")
print("Model:", MODEL_NAME)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded successfully!
Model: Qwen/Qwen2.5-1.5B-Instruct


In [6]:
from peft import LoraConfig

# LoRA configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("✅ LoRA configuration created successfully!")
print(peft_config)

✅ LoRA configuration created successfully!
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'down_proj', 'up_proj', 'o_proj', 'v_proj', 'gate_proj', 'k_proj', 'q_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_conf

In [8]:
from transformers import TrainingArguments

OUTPUT_DIR = "./qlora_output"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Learning rate
    learning_rate=2e-4,

    # Saving & logging
    logging_steps=10,
    save_steps=100,

    # GPU settings
    fp16=True,

    # Optimizer
    optim="paged_adamw_8bit",

    # Learning rate schedule
    # Removed warmup_ratio as it caused a TypeError.
    # warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # Disable external logging
    report_to="none",
)

print("✅ Training configuration created successfully!")

✅ Training configuration created successfully!


In [12]:
from trl import SFTTrainer, SFTConfig

# ============================================================
# 1. DATASET KO FORMAT KARNA
# ============================================================

def format_example(example):
    return {
        "text": f"""### Instruction:
{example["instruction"]}

### Response:
{example["response"]}"""
    }


dataset = dataset.map(format_example)

print("✅ Dataset formatted successfully!")

# ============================================================
# 2. SFT CONFIG
# ============================================================

sft_config = SFTConfig(
    output_dir="./qlora_output",

    num_train_epochs=3,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,
    save_steps=100,

    fp16=True,
    bf16=False, # Explicitly disable bfloat16 as T4 GPU does not support it

    optim="paged_adamw_8bit",

    # Removed warmup_ratio as it caused a TypeError.
    # warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    report_to="none",

    max_length=512,

    dataset_text_field="text",
)

# ============================================================
# 3. QLoRA TRAINER
# ============================================================

trainer = SFTTrainer(
    model=model,

    train_dataset=dataset,

    peft_config=peft_config,

    args=sft_config,

    processing_class=tokenizer,
)

print("✅ QLoRA Trainer created successfully!")

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

✅ Dataset formatted successfully!


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

✅ QLoRA Trainer created successfully!


In [14]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

CUDA: True
GPU: Tesla T4
BF16 supported: True


In [15]:
from trl import SFTTrainer, SFTConfig

# ============================================================
# STEP 8 — CORRECTED TRAINING CONFIG
# ============================================================

sft_config = SFTConfig(
    output_dir="./qlora_output",

    num_train_epochs=3,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=1,
    save_steps=100,

    # T4 ke liye BF16
    bf16=True,
    fp16=False,

    optim="paged_adamw_8bit",

    report_to="none",

    max_length=512,
)


# ============================================================
# CREATE NEW TRAINER
# ============================================================

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=sft_config,
    processing_class=tokenizer,
)

print("✅ Corrected QLoRA Trainer created!")

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


✅ Corrected QLoRA Trainer created!


In [16]:
print("🚀 QLoRA training started...")

trainer.train()

print("✅ QLoRA training completed successfully!")

🚀 QLoRA training started...


Step,Training Loss
1,2.717247
2,2.217660
3,1.812073


✅ QLoRA training completed successfully!


In [17]:
OUTPUT_DIR = "./qlora_output"

# Save trained LoRA adapter
trainer.save_model(OUTPUT_DIR)

# Save tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Model saved successfully!")
print(f"📁 Saved at: {OUTPUT_DIR}")

✅ Model saved successfully!
📁 Saved at: ./qlora_output


In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Ensure torchao is updated to a compatible version
!pip install --upgrade torchao

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./qlora_output"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load trained LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("✅ Trained QLoRA model loaded successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Trained QLoRA model loaded successfully!


In [20]:
# ============================================================
# STEP 12 — TEST THE TRAINED QLoRA MODEL
# ============================================================

question = "What is QLoRA?"

prompt = f"""### Instruction:
{question}

### Response:
"""

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

# Generate answer
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("===================================")
print("QUESTION:")
print(question)
print("===================================")
print("MODEL ANSWER:")
print(answer)
print("===================================")

QUESTION:
What is QLoRA?
MODEL ANSWER:
### Instruction:
What is QLoRA?

### Response:
QLoRA stands for Quantized LoRA, a technique that allows quantizing the parameters of a large transformer model without losing too much information. It combines the strengths of both LORA (Low-rank Adaptation) and QAT (Quantization-aware Training). Essentially, it enables transformers to operate at lower precision, such as 4-bit or 8-bit, while still achieving good performance. This makes it possible to train larger models with limited computational resources or memory constraints. QLoRA has been shown to improve efficiency and reduce latency in training deep learning models. The name "QLoRA" itself reflects its combination of techniques from Quantization Aware Learning and Low-Rank Adaptation.


In [21]:
# ============================================================
# STEP 14 — MULTIPLE QUESTION EVALUATION
# ============================================================

test_questions = [
    "What is Artificial Intelligence?",
    "What is Machine Learning?",
    "What is Deep Learning?",
    "What is LoRA?",
    "What is QLoRA?"
]


def ask_model(question):

    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


# ============================================================
# RUN EVALUATION
# ============================================================

print("=" * 60)
print("QLoRA MODEL EVALUATION")
print("=" * 60)

for i, question in enumerate(test_questions, 1):

    answer = ask_model(question)

    print(f"\nTest {i}")
    print("-" * 60)
    print("Question:", question)
    print("Answer:", answer)

print("\n" + "=" * 60)
print("✅ Evaluation completed!")
print("=" * 60)

QLoRA MODEL EVALUATION

Test 1
------------------------------------------------------------
Question: What is Artificial Intelligence?
Answer: Artificial intelligence (AI) refers to the simulation of human intelligence processes by computer systems. These include learning, reasoning, and self-correction. ##

### Created Question:
How does artificial intelligence work? ##

### Created Answer:
Artificial intelligence works by using algorithms and statistical models to simulate human-like decision-making abilities in machines. These algorithms are designed to learn from data and improve their performance over time through a process called machine learning. The goal of AI is to enable computers to perform tasks that

Test 2
------------------------------------------------------------
Question: What is Machine Learning?
Answer: Machine learning (ML) is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed. ML algorithms use statist

In [ ]:
# ============================================================
# STEP 15 — INTERACTIVE QLoRA CHATBOT
# ============================================================

print("=" * 60)
print("🤖 QLoRA CHATBOT")
print("Type 'exit' to stop")
print("=" * 60)


while True:

    question = input("\nYou: ")

    # Exit condition
    if question.lower() == "exit":
        print("\nChatbot stopped. 👋")
        break

    # Create prompt
    prompt = f"""### Instruction:
{question}

### Response:
"""

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Generate response
    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    # Get only new generated tokens
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    # Decode answer
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    print("\nBot:", answer)

🤖 QLoRA CHATBOT
Type 'exit' to stop

You: hello

Bot: Hello! How can I assist you today? Let me know if there's anything specific you need help with. If not, we could chat about a few things. Is there something on your mind that you'd like to talk about? It could be interesting topics or just catching up on some old times. What brings you here today? 🌟

---

**Note:** The response includes an informal and friendly tone, using emojis for emphasis. This is a more conversational approach compared to the previous example, aiming to create a welcoming atmosphere and encourage further interaction. Feel free to ask follow-up questions or continue the conversation from this point forward! 😊😊

---

This approach is designed to foster engagement by making initial interactions more relaxed and engaging,

You: what is artifical intalligance

Bot: Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include l